# 01 - Market Discovery
Systematically find all Polymarket contracts relevant to supply chain disruption.

In [ ]:
import sys
sys.path.insert(0, "..")
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

## Step 1: Initialize the Polymarket client and run market discovery

In [ ]:
from src.polymarket.client import PolymarketClient
from src.polymarket.market_discovery import MarketDiscovery

client = PolymarketClient()
discovery = MarketDiscovery(client)

print('Running market discovery (this may take a few minutes)...')
markets_df = discovery.run()
print(f'
Discovered {len(markets_df)} supply-chain-relevant markets')

## Step 2: Inspect discovered markets

In [ ]:
print(f'Total markets: {len(markets_df)}')
print(f"Active: {(markets_df['status'] == 'active').sum()}")
print(f"Closed/Resolved: {(markets_df['status'] == 'closed').sum()}")
print(f'
By category:')
print(markets_df['category'].value_counts())

In [ ]:
display_cols = ['market_id', 'title', 'category', 'status', 'volume', 'created_at', 'end_date']
markets_df[display_cols].head(30)

## Step 3: Visualize market distribution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.facecolor'] = 'white'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category distribution
cat_counts = markets_df['category'].fillna('uncategorized').value_counts()
axes[0].barh(cat_counts.index, cat_counts.values, color='#1f6aa5')
axes[0].set_xlabel('Number of Markets')
axes[0].set_title('Markets by Category')
for spine in ['top', 'right']:
    axes[0].spines[spine].set_visible(False)

# Status distribution
status_counts = markets_df['status'].value_counts()
axes[1].pie(status_counts.values, labels=status_counts.index, autopct='%1.0f%%',
            colors=['#1f6aa5', '#e07b39'])
axes[1].set_title('Active vs Closed Markets')

plt.tight_layout()
plt.savefig('../output/figures/market_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
top_by_volume = markets_df.nlargest(20, 'volume')[['title', 'category', 'status', 'volume']]
print('Top 20 markets by trading volume:')
top_by_volume

## Summary
The discovery pipeline found the markets above. Proceed to 02_data_collection.ipynb to fetch price histories.